# MSD Special-State Physical Prototype

This notebook prototypes the paper-faithful MSD special-state construction using the `distillation_sim` reference implementation.

Target construction:
- apply the basis-dependent tomography change on the distinguished input qubit
- apply the inverse of the MSD circuit on the physical 5-qubit input state
- encode each input qubit into a Steane logical block
- run the logical MSD circuit

This notebook intentionally does **not** modify `demo/msd_utils/circuits.py` yet. The goal here is to verify the reference behavior first.

Important interpretation:
- `distillation_sim` separates the ideal logical action `C` from the physical noise channel `E_sigma`
- for decoder training, we care about the **physical** detector/observable channel coming from `physical_circ.sample_detectors_and_observables(...)`
- in noiseless simulation, that physical channel should produce detector bits `0...0` and logical observable bits `0...0`
- the combined sampler `sample_det_obs_shots(...)` additionally XORs in the ideal logical measurement outcomes, so those combined observables are not expected to be all zero

In [1]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import HTML, display

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in REPO_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate the bloqade-lanes repo root.')

DISTILLATION_SIM_CANDIDATES = [
    Path('/Users/jasonhan/Desktop/qmain/personal-workspace/distillation_sim'),
    REPO_ROOT.parent.parent / 'personal-workspace' / 'distillation_sim',
]
for candidate in DISTILLATION_SIM_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'src' / 'distillation_sim').exists():
        DISTILLATION_SIM_ROOT = candidate
        sys.path.insert(0, str(candidate / 'src'))
        break
else:
    raise FileNotFoundError('Could not locate the distillation_sim repo.')

from distillation_sim.circuits.noise_model import msd_updated_noise_model
from distillation_sim.circuits.steane import SteaneCircuit
from distillation_sim.simulation.sampling import CombinedSampler

print('bloqade-lanes:', REPO_ROOT)
print('distillation_sim:', DISTILLATION_SIM_ROOT)

Set parameter LogFile to value "gurobi_decoder.log"
Restricted license - for non-production use only - expires 2027-11-29
bloqade-lanes: /Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes
distillation_sim: /Users/jasonhan/Desktop/qmain/personal-workspace/distillation_sim


In [2]:
BASIS_LABELS = ('X', 'Y', 'Z')


def make_lossless_steane_factory(num_logical_qubits: int = 5) -> SteaneCircuit:
    noise = msd_updated_noise_model(
        num_logical_qubits=num_logical_qubits,
        patch_size=7,
        global_scalar=0.0,
        loss_scalar=0.0,
        heat_scalar=0.0,
        new_bb1=True,
        new_1q=True,
        new_xy=True,
        new_cz=True,
        new_move=True,
    )
    return SteaneCircuit(noise, num_logical_qubits=num_logical_qubits)


def lossless_stim_circuit(color_code_circuit):
    return color_code_circuit.noisy_circuit.circuit.get_lossless_circuit()


def make_lossless_sampler(basis: str) -> CombinedSampler:
    return CombinedSampler.msd_circuit(
        basis,
        inject_error_prob=0.0,
        error_channel='Zrot',
        distance=3,
        add_echoes=True,
        moving_pattern='to_left',
        global_scalar=0.0,
        apply_loss=False,
        new_bb1=True,
        new_1q=True,
        new_xy=True,
        new_cz=True,
        new_move=True,
    )


def summarize_shots(detectors: np.ndarray, observables: np.ndarray) -> dict[str, object]:
    return {
        'unique_detectors': np.unique(detectors, axis=0).tolist(),
        'unique_observables': np.unique(observables, axis=0).tolist(),
        'all_zero_detectors': bool(np.all(detectors == 0)),
        'all_zero_observables': bool(np.all(observables == 0)),
    }


def sample_lossless_physical_stim_circuit(basis: str, shots: int = 256, seed: int = 1) -> dict[str, object]:
    sampler = make_lossless_sampler(basis)
    lossless = sampler.physical_circ.noisy_circuit.circuit.get_lossless_circuit()
    det, obs = lossless.compile_detector_sampler(seed=seed).sample(
        shots=shots,
        separate_observables=True,
    )
    return summarize_shots(det, obs)


def show_svg(stim_circuit, *, height: int = 360) -> None:
    svg_text = str(stim_circuit.diagram('timeline-svg'))
    svg_text = svg_text.replace(
        '<svg ',
        f'<svg style="height:{height}px; width:auto; max-width:none; display:block;" ',
        1,
    )
    display(
        HTML(
            '<div style="overflow-x:auto; width:100%; border:1px solid #ddd; padding:8px; background:white;">'
            + svg_text
            + '</div>'
        )
    )


def build_stage_circuits(basis: str = 'X') -> dict[str, object]:
    fake_input = make_lossless_steane_factory()
    fake_input.append_naive_layout()
    fake_input.sh_fake_input_circuit(basis)

    fake_input_plus_encoding = make_lossless_steane_factory()
    fake_input_plus_encoding.append_naive_layout()
    fake_input_plus_encoding.sh_fake_input_circuit(basis)
    fake_input_plus_encoding.append_encoding_circuit()

    full_sampler = make_lossless_sampler(basis)

    return {
        'fake_input': lossless_stim_circuit(fake_input),
        'fake_input_plus_encoding': lossless_stim_circuit(fake_input_plus_encoding),
        'full_msd': full_sampler.physical_circ.noisy_circuit.circuit.get_lossless_circuit(),
    }


## Verify the physical channel is deterministic in noiseless simulation

This is the reference check that matters for decoder-training labels. We sample from the **physical** circuit only, without XORing in the ideal logical circuit outcomes.

### Directly sample the exact lossless Stim circuit

This samples the exact lossless physical circuit returned by `distillation_sim` and checks whether its detector and observable outputs are all zero.

In [3]:
LOSSLESS_STIM_RESULTS = {
    basis: sample_lossless_physical_stim_circuit(basis)
    for basis in BASIS_LABELS
}

LOSSLESS_STIM_RESULTS

{'X': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Y': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Z': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True}}

In [4]:
PHYSICAL_RESULTS = {}
for basis in BASIS_LABELS:
    sampler = make_lossless_sampler(basis)
    det, obs = sampler.physical_circ.sample_detectors_and_observables(num_shots=256, seed=1)
    PHYSICAL_RESULTS[basis] = summarize_shots(det, obs)

PHYSICAL_RESULTS

{'X': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Y': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Z': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False]],
  'all_zero_detectors': True,
  'all_zero_observables': True}}

In [5]:
LOSSLESS_CONSISTENCY = {
    basis: {
        'direct_stim_matches_physical_wrapper': LOSSLESS_STIM_RESULTS[basis] == PHYSICAL_RESULTS[basis],
    }
    for basis in BASIS_LABELS
}

LOSSLESS_CONSISTENCY

{'X': {'direct_stim_matches_physical_wrapper': True},
 'Y': {'direct_stim_matches_physical_wrapper': True},
 'Z': {'direct_stim_matches_physical_wrapper': True}}

## Compare with the combined logical-plus-physical sampler

This sampler includes the ideal logical circuit outcomes, so the observables are not expected to be all zero even in the noiseless limit.

In [6]:
COMBINED_RESULTS = {}
for basis in BASIS_LABELS:
    sampler = make_lossless_sampler(basis)
    det, obs = sampler.sample_det_obs_shots(num_shots=256, seed=1)
    COMBINED_RESULTS[basis] = summarize_shots(det, obs)

COMBINED_RESULTS

{'X': {'unique_detectors': [[False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False,
    False]],
  'unique_observables': [[False, False, False, False, False],
   [False, False, False, False, True],
   [False, False, False, True, False],
   [False, False, False, True, True],
   [False, False, True, False, False],
   [False, False, True, False, True],
   [False, False, True, True, False],
   [False, False, True, True, True],
   [False, True, False, False, False],
   [False, True, False, True, False],
   [False, True, False, True, True],
   [False, True, True, False, False],
   [False, True, True, False, True],
   [False, True, True, True, False],
   [False, True, True, True, True],
   [True, False, False, False, True],
   [True, False, False, True, False],
   [True, False, False, True, True],
   [True, False, True, False, False],
   [True, False, True, False, True],
   [True, False, True, True,

## Visualize the staged construction for one basis

This mirrors the intended special-state recipe:
- fake input on the five distinguished physical qubits
- fake input plus Steane encoding
- full lossless MSD physical circuit

In [7]:
BASIS = 'X'
stage_circuits = build_stage_circuits(BASIS)
special_input_qubits = [i * 7 + 6 for i in range(5)]
print('basis =', BASIS)
print('special input qubits =', special_input_qubits)

basis = X
special input qubits = [6, 13, 20, 27, 34]


In [8]:
show_svg(stage_circuits['fake_input'])

In [7]:
show_svg(stage_circuits['fake_input_plus_encoding'])

In [9]:
show_svg(stage_circuits['full_msd'], height=1000)

## Integration note for `bloqade-lanes`

The current `bloqade-lanes` logical initialization hook initializes each future logical block independently. That is enough for single-block special states, but not enough to directly express a 5-qubit entangled fake input across blocks before Steane encoding. This notebook therefore serves as the paper-side reference behavior we want to match before deciding how to integrate it into `demo/msd_utils/circuits.py`.